In [10]:
import glob
import pandas as pd
from helper import *
from torch.optim import Adam
from torchinfo import summary
from torch.utils.data import Dataset, DataLoader

I moved classes and functions from the previous notebook to the `helper.py` not to rewrite the same code

In [11]:
HR_train_paths = sorted(glob.glob("../data/DIV2K_train_HR/*.png"))
X2_train_paths = sorted(glob.glob("../data/DIV2K_train_LR_bicubic/X2/*.png"))
X4_train_paths = sorted(glob.glob("../data/DIV2K_train_LR_bicubic/X4/*.png"))
X8_train_paths = sorted(glob.glob("../data/DIV2K_train_LR_bicubic/X8/*.png"))
X16_train_paths = sorted(glob.glob("../data/DIV2K_train_LR_bicubic/X16/*.png"))
X32_train_paths = sorted(glob.glob("../data/DIV2K_train_LR_bicubic/X32/*.png"))
X64_train_paths = sorted(glob.glob("../data/DIV2K_train_LR_bicubic/X64/*.png"))

HR_valid_paths = sorted(glob.glob("../data/DIV2K_valid_HR/*.png"))
X2_valid_paths = sorted(glob.glob("../data/DIV2K_valid_LR_bicubic/X2/*.png"))
X4_valid_paths = sorted(glob.glob("../data/DIV2K_valid_LR_bicubic/X4/*.png"))
X8_valid_paths = sorted(glob.glob("../data/DIV2K_valid_LR_bicubic/X8/*.png"))
X16_valid_paths = sorted(glob.glob("../data/DIV2K_valid_LR_bicubic/X16/*.png"))
X32_valid_paths = sorted(glob.glob("../data/DIV2K_valid_LR_bicubic/X32/*.png"))
X64_valid_paths = sorted(glob.glob("../data/DIV2K_valid_LR_bicubic/X64/*.png"))

## X4 Scaling

In [12]:
# checkpoint = torch.load('./model_checkpoints/EDSR/X2.pth')
model = EDSR_Light(2).to(device)
# model.load_state_dict(checkpoint['model_state_dict'])
model.upscaling_head

Sequential(
  (0): Conv2d(64, 256, kernel_size=(3, 3), stride=(1, 1), padding=same)
  (1): PixelShuffle(upscale_factor=2)
  (2): PReLU(num_parameters=1)
  (3): Conv2d(64, 3, kernel_size=(9, 9), stride=(1, 1), padding=same)
)

In [4]:
# remove the last convolutional layer
model.upscaling_head = model.upscaling_head[:-1]
model.upscaling_head

Sequential(
  (0): Conv2d(64, 256, kernel_size=(3, 3), stride=(1, 1), padding=same)
  (1): PixelShuffle(upscale_factor=2)
  (2): PReLU(num_parameters=1)
)

In [5]:
head = nn.Sequential(
    nn.Conv2d(64, 256, 3, stride=1, padding='same'),
    nn.PixelShuffle(2),
    nn.PReLU(),
    nn.Conv2d(64, 3, 9, stride=1, padding='same')
)

In [6]:
model.upscaling_head = nn.Sequential(*model.upscaling_head, *head)
model.upscaling_head

Sequential(
  (0): Conv2d(64, 256, kernel_size=(3, 3), stride=(1, 1), padding=same)
  (1): PixelShuffle(upscale_factor=2)
  (2): PReLU(num_parameters=1)
  (3): Conv2d(64, 256, kernel_size=(3, 3), stride=(1, 1), padding=same)
  (4): PixelShuffle(upscale_factor=2)
  (5): PReLU(num_parameters=1)
  (6): Conv2d(64, 3, kernel_size=(9, 9), stride=(1, 1), padding=same)
)

In [7]:
model.upscaling_head[:-4]

Sequential(
  (0): Conv2d(64, 256, kernel_size=(3, 3), stride=(1, 1), padding=same)
  (1): PixelShuffle(upscale_factor=2)
  (2): PReLU(num_parameters=1)
)

For the first 1000 epochs I freeze the pretrained layers

In [8]:
for param in model.expand.parameters():
    param.requires_grad = False

for param in model.residual_blocks.parameters():
    param.requires_grad = False

for param in model.upscaling_head[:-4].parameters():
    param.requires_grad = False

In [9]:
model.upscaling_head[-4:]

Sequential(
  (3): Conv2d(64, 256, kernel_size=(3, 3), stride=(1, 1), padding=same)
  (4): PixelShuffle(upscale_factor=2)
  (5): PReLU(num_parameters=1)
  (6): Conv2d(64, 3, kernel_size=(9, 9), stride=(1, 1), padding=same)
)

In [62]:
summary(model, col_names=['trainable'])

In [ ]:
valid_ds = EDSR_Dataset(HR_valid_paths, 4, ram_limit_gb=1)

In [ ]:
train_ds = EDSR_Dataset(HR_train_paths, 4, ram_limit_gb=8)

In [ ]:
train_dl = DataLoader(train_ds, batch_size=16, shuffle=True, num_workers=os.cpu_count()-1)
valid_dl = DataLoader(valid_ds, batch_size=16, shuffle=False, num_workers=os.cpu_count()-1)

loss_fn = nn.L1Loss()
optimizer = Adam(model.upscaling_head[-4:].parameters(), lr=1e-4)
scheduler = StepLR(optimizer, step_size=200, gamma=0.5)
model = troch.compile(model)

train(model, train_dl, valid_dl, optimizer, scheduler, loss_fn, 1000)

In [ ]:
for param in model.expand.parameters():
    param.requires_grad = True

for param in model.residual_blocks.parameters():
    param.requires_grad = True

for param in model.upscaling_head[:-4].parameters():
    param.requires_grad = True

In [ ]:
optimizer = Adam(model.parameters(), lr=1e-5)
scheduler = StepLR(optimizer, step_size=1000, gamma=0.5)

train(model, train_dl, valid_dl, optimizer, scheduler, loss_fn, 5000)

In [ ]:
checkpoint = torch.load('../model_checkpoints/EDSR_Light/X4.pth')
model = EDSR_Light(4).to(device)
model.load_state_dict(checkpoint['model_state_dict'])

In [ ]:
model.eval()

In [ ]:
targets = [X16_valid_paths, X8_valid_paths, X4_valid_paths, X2_valid_paths, HR_valid_paths]

metrics_x4 = pd.DataFrame(columns=["PSNR↑", "SSIM↑", "LPIPS↓"])
for target_ds in tqdm(targets, total=5):
    metrics_x4.loc[len(metrics_x4)] = calc_metrics(model, target_ds, 4)

metrics_x4.index = [
    "31px -> 124px", "63px -> 252px", "127px -> 508px", 
    "255px -> 1020px", "510px -> 2040px"
]
metrics_x4

### Super-resolution showcase

In [ ]:
print("31px -> 124px")
inp = transform(Image.open(X64_valid_paths[4])).to(device)
with torch.inference_mode():
    out = model(inp).clamp(0.0, 1.0) * 255.0

img = Image.fromarray(out.permute(1, 2, 0).to(torch.uint8).cpu().numpy())
os.makedirs('../image_results/4X/31px', exist_ok=True)
img.save('../image_results/4X/31px/EDSR_Light_31px.png')
img

In [ ]:
print("63px -> 252px")
inp = transform(Image.open(X32_valid_paths[4])).to(device)
with torch.inference_mode():
    out = model(inp).clamp(0.0, 1.0) * 255.0

img = Image.fromarray(out.permute(1, 2, 0).to(torch.uint8).cpu().numpy())
os.makedirs('../image_results/4X/63px', exist_ok=True)
img.save('../image_results/4X/63px/EDSR_Light_63px.png')
img

In [ ]:
print("127px -> 508px")
inp = transform(Image.open(X16_valid_paths[4])).to(device)
with torch.inference_mode():
    out = model(inp).clamp(0.0, 1.0) * 255.0

img = Image.fromarray(out.permute(1, 2, 0).to(torch.uint8).cpu().numpy())
os.makedirs('../image_results/4X/127px', exist_ok=True)
img.save('../image_results/4X/127px/EDSR_Light_127px.png')
img

In [ ]:
print("255px -> 1020px")
inp = transform(Image.open(X8_valid_paths[4])).to(device)
with torch.inference_mode():
    out = model(inp).clamp(0.0, 1.0) * 255.0

img = Image.fromarray(out.permute(1, 2, 0).to(torch.uint8).cpu().numpy())
os.makedirs('../image_results/4X/255px', exist_ok=True)
img.save('../image_results/4X/255px/EDSR_Light_255px.png')
img

In [ ]:
print("510px -> 2040px")
inp = transform(Image.open(X4_valid_paths[4])).to(device)
with torch.inference_mode():
    out = model(inp).clamp(0.0, 1.0) * 255.0

img = Image.fromarray(out.permute(1, 2, 0).to(torch.uint8).cpu().numpy())
os.makedirs('../image_results/4X/510px', exist_ok=True)
img.save('../image_results/4X/510px/EDSR_Light_510px.png')
img

## Geometric Self-ensemble X4

In [ ]:
class GSE:
    def __init__(self, model):
        self.model = model
        self.rotations = [0, 90, 180, 270]

        self.transforms = v2.Compose([
            v2.PILToTensor(),
            v2.Lambda(lambda x: x / 255.0)
        ])

    def __call__(self, x):
        prediction = torch.zeros(self.model(x).shape).to(device)
        for rotation in self.rotations:
            rot = torch.rot90(x, k=rotation // 90, dims=[-2, -1])
            for i in range(2):
                with torch.inference_mode():
                    if i:
                        flip = v2.functional.horizontal_flip(rot)
                        out = torch.flip(self.model(flip), dims=[-1])
                    else:
                        out = self.model(rot)

                prediction += torch.rot90(out, k=4 - rotation // 90, dims=[-2, -1])

        return prediction / 8.0

In [ ]:
model_gse = GSE(model)

In [ ]:
targets = [X16_valid_paths, X8_valid_paths, X4_valid_paths, X2_valid_paths, HR_valid_paths]

metrics_x4_gse = pd.DataFrame(columns=["PSNR↑", "SSIM↑", "LPIPS↓"])
for target_ds in tqdm(targets, total=5):
    metrics_x4_gse.loc[len(metrics_x4)] = calc_metrics(model, target_ds, 4)

metrics_x4_gse.index = [
    "31px -> 124px", "63px -> 252px", "127px -> 508px", 
    "255px -> 1020px", "510px -> 2040px"
]
metrics_x4_gse

### Super-resolution showcase

In [ ]:
print("31px -> 124px")
inp = transform(Image.open(X64_valid_paths[4])).to(device)
with torch.inference_mode():
    out = model(inp).clamp(0.0, 1.0) * 255.0

img = Image.fromarray(out.permute(1, 2, 0).to(torch.uint8).cpu().numpy())
os.makedirs('../image_results/4X/31px', exist_ok=True)
img.save('../image_results/4X/31px/EDSR_Light+_31px.png')
img

In [ ]:
print("63px -> 252px")
inp = transform(Image.open(X32_valid_paths[4])).to(device)
with torch.inference_mode():
    out = model(inp).clamp(0.0, 1.0) * 255.0

img = Image.fromarray(out.permute(1, 2, 0).to(torch.uint8).cpu().numpy())
os.makedirs('../image_results/4X/63px', exist_ok=True)
img.save('../image_results/4X/63px/EDSR_Light+_63px.png')
img

In [ ]:
print("127px -> 508px")
inp = transform(Image.open(X16_valid_paths[4])).to(device)
with torch.inference_mode():
    out = model(inp).clamp(0.0, 1.0) * 255.0

img = Image.fromarray(out.permute(1, 2, 0).to(torch.uint8).cpu().numpy())
os.makedirs('../image_results/4X/127px', exist_ok=True)
img.save('../image_results/4X/127px/EDSR_Light+_127px.png')
img

In [ ]:
print("255px -> 1020px")
inp = transform(Image.open(X8_valid_paths[4])).to(device)
with torch.inference_mode():
    out = model(inp).clamp(0.0, 1.0) * 255.0

img = Image.fromarray(out.permute(1, 2, 0).to(torch.uint8).cpu().numpy())
os.makedirs('../image_results/4X/255px', exist_ok=True)
img.save('../image_results/4X/255px/EDSR_Light+_255px.png')
img

In [ ]:
print("510px -> 2040px")
inp = transform(Image.open(X4_valid_paths[4])).to(device)
with torch.inference_mode():
    out = model(inp).clamp(0.0, 1.0) * 255.0

img = Image.fromarray(out.permute(1, 2, 0).to(torch.uint8).cpu().numpy())
os.makedirs('../image_results/4X/510px', exist_ok=True)
img.save('../image_results/4X/510px/EDSR_Light+_510px.png')
img

## X8 Scaling

The process of preparation for training is the same as above, simply the starting point is the **4X model**, not the 2X model

In [64]:
# checkpoint = torch.load('./model_checkpoints/EDSR/X4.pth')
model = EDSR_Light(4).to(device)
# model.load_state_dict(checkpoint['model_state_dict'])

model.upscaling_head = model.upscaling_head[:-1]

head = nn.Sequential(
    nn.Conv2d(64, 256, 3, stride=1, padding='same'),
    nn.PixelShuffle(2),
    nn.PReLU(),
    nn.Conv2d(64, 3, 9, stride=1, padding='same')
)

model.upscaling_head = nn.Sequential(*model.upscaling_head, *head)

for param in model.expand.parameters():
    param.requires_grad = False

for param in model.residual_blocks.parameters():
    param.requires_grad = False

for param in model.upscaling_head[:-4].parameters():
    param.requires_grad = False

# summary(model, col_names=['trainable'])

Layer (type:depth-idx)                   Trainable
EDSR_Light                               Partial
├─Sequential: 1-1                        False
│    └─Conv2d: 2-1                       False
│    └─PReLU: 2-2                        False
├─Sequential: 1-2                        False
│    └─ResBlockEDSRLight: 2-3            False
│    │    └─Sequential: 3-1              False
│    └─ResBlockEDSRLight: 2-4            False
│    │    └─Sequential: 3-2              False
│    └─ResBlockEDSRLight: 2-5            False
│    │    └─Sequential: 3-3              False
│    └─ResBlockEDSRLight: 2-6            False
│    │    └─Sequential: 3-4              False
│    └─ResBlockEDSRLight: 2-7            False
│    │    └─Sequential: 3-5              False
│    └─ResBlockEDSRLight: 2-8            False
│    │    └─Sequential: 3-6              False
│    └─ResBlockEDSRLight: 2-9            False
│    │    └─Sequential: 3-7              False
│    └─ResBlockEDSRLight: 2-10           False
│    │ 

In [ ]:
valid_ds = EDSR_Dataset(HR_valid_paths, 2, ram_limit_gb=1)

In [ ]:
train_ds = EDSR_Dataset(HR_train_paths, 2, ram_limit_gb=8)

In [ ]:
train_dl = DataLoader(train_ds, batch_size=16, shuffle=True, num_workers=os.cpu_count()-1)
valid_dl = DataLoader(valid_ds, batch_size=16, shuffle=False, num_workers=os.cpu_count()-1)

loss_fn = nn.L1Loss()
optimizer = Adam(model.upscaling_head[-4:].parameters(), lr=1e-4)
scheduler = StepLR(optimizer, step_size=200, gamma=0.5)
model = torch.compile(model)

train(model, train_dl, valid_dl, optimizer, scheduler, loss_fn, 1000)

In [ ]:
for param in model.expand.parameters():
    param.requires_grad = True

for param in model.residual_blocks.parameters():
    param.requires_grad = True

for param in model.upscaling_head[:-4].parameters():
    param.requires_grad = True

In [ ]:
optimizer = Adam(model.parameters(), lr=1e-5)
scheduler = StepLR(optimizer, step_size=1000, gamma=0.5)

train(model, train_dl, valid_dl, optimizer, scheduler, loss_fn, 5000)

In [ ]:
checkpoint = torch.load('../model_checkpoints/EDSR_Light/X8.pth')
model = EDSR_Light(8).to(device)
model.load_state_dict(checkpoint['model_state_dict'])

In [ ]:
model.eval()

In [ ]:
targets = [X8_valid_paths, X4_valid_paths, X2_valid_paths, HR_valid_paths]

metrics_x8 = pd.DataFrame(columns=["PSNR↑", "SSIM↑", "LPIPS↓"])
for target_ds in tqdm(targets, total=4):
    metrics_x8.loc[len(metrics_x8)] = calc_metrics(model, target_ds, 8)

metrics_x8.index = ["31px -> 248px", "63px -> 504px", "127px -> 1016px", "255px -> 2040px"]
metrics_x8

### Super-resolution showcase

In [ ]:
print("31px -> 248px")
inp = transform(Image.open(X64_valid_paths[4])).to(device)
with torch.inference_mode():
    out = model(inp).clamp(0.0, 1.0) * 255.0

img = Image.fromarray(out.permute(1, 2, 0).to(torch.uint8).cpu().numpy())
os.makedirs('../image_results/8X/31px', exist_ok=True)
img.save('../image_results/8X/31px/EDSR_Light_31px.png')
img

In [ ]:
print("63px -> 504px")
inp = transform(Image.open(X32_valid_paths[4])).to(device)
with torch.inference_mode():
    out = model(inp).clamp(0.0, 1.0) * 255.0

img = Image.fromarray(out.permute(1, 2, 0).to(torch.uint8).cpu().numpy())
os.makedirs('../image_results/8X/63px', exist_ok=True)
img.save('../image_results/8X/63px/EDSR_Light_63px.png')
img

In [ ]:
print("127px -> 1016px")
inp = transform(Image.open(X16_valid_paths[4])).to(device)
with torch.inference_mode():
    out = model(inp).clamp(0.0, 1.0) * 255.0

img = Image.fromarray(out.permute(1, 2, 0).to(torch.uint8).cpu().numpy())
os.makedirs('../image_results/8X/127px', exist_ok=True)
img.save('../image_results/8X/127px/EDSR_Light_127px.png')
img

In [ ]:
print("255px -> 2040px")
inp = transform(Image.open(X8_valid_paths[4])).to(device)
with torch.inference_mode():
    out = model(inp).clamp(0.0, 1.0) * 255.0

img = Image.fromarray(out.permute(1, 2, 0).to(torch.uint8).cpu().numpy())
os.makedirs('../image_results/8X/255px', exist_ok=True)
img.save('../image_results/8X/255px/EDSR_Light_255px.png')
img

## Geometric Self-ensemble X8

In [ ]:
model_gse = GSE(model)

In [ ]:
targets = [X8_valid_paths, X4_valid_paths, X2_valid_paths, HR_valid_paths]

metrics_x8_gse = pd.DataFrame(columns=["PSNR↑", "SSIM↑", "LPIPS↓"])
for target_ds in tqdm(targets, total=4):
    metrics_x8_gse.loc[len(metrics_x8_gse)] = calc_metrics(model, target_ds, 8)

metrics_x8_gse.index = ["31px -> 248px", "63px -> 504px", "127px -> 1016px", "255px -> 2040px"]
metrics_x8_gse

### Super-resolution showcase

In [ ]:
print("31px -> 248px")
inp = transform(Image.open(X64_valid_paths[4])).to(device)
with torch.inference_mode():
    out = model(inp).clamp(0.0, 1.0) * 255.0

img = Image.fromarray(out.permute(1, 2, 0).to(torch.uint8).cpu().numpy())
os.makedirs('../image_results/8X/31px', exist_ok=True)
img.save('../image_results/8X/31px/EDSR_Light_31px.png')
img

In [ ]:
print("63px -> 504px")
inp = transform(Image.open(X32_valid_paths[4])).to(device)
with torch.inference_mode():
    out = model(inp).clamp(0.0, 1.0) * 255.0

img = Image.fromarray(out.permute(1, 2, 0).to(torch.uint8).cpu().numpy())
os.makedirs('../image_results/8X/63px', exist_ok=True)
img.save('../image_results/8X/63px/EDSR_Light_63px.png')
img

In [ ]:
print("127px -> 1016px")
inp = transform(Image.open(X16_valid_paths[4])).to(device)
with torch.inference_mode():
    out = model(inp).clamp(0.0, 1.0) * 255.0

img = Image.fromarray(out.permute(1, 2, 0).to(torch.uint8).cpu().numpy())
os.makedirs('../image_results/8X/127px', exist_ok=True)
img.save('../image_results/8X/127px/EDSR_Light_127px.png')
img

In [ ]:
print("255px -> 2040px")
inp = transform(Image.open(X8_valid_paths[4])).to(device)
with torch.inference_mode():
    out = model(inp).clamp(0.0, 1.0) * 255.0

img = Image.fromarray(out.permute(1, 2, 0).to(torch.uint8).cpu().numpy())
os.makedirs('../image_results/8X/255px', exist_ok=True)
img.save('../image_results/8X/255px/EDSR_Light_255px.png')
img

## Final Results

In [ ]:
checkpoint = torch.load('../model_checkpoints/EDSR_Light/X2.pth')
model = EDSR_Light(2).to(device)
model.load_state_dict(checkpoint['model_state_dict'])

In [ ]:
model.eval();

In [ ]:
targets = [X32_valid_paths, X16_valid_paths, X8_valid_paths, X4_valid_paths, X2_valid_paths, HR_valid_paths]

metrics_x2 = pd.DataFrame(columns=["PSNR↑", "SSIM↑", "LPIPS↓"])
for target_ds in tqdm(targets, total=6):
    metrics_x2.loc[len(metrics_x2)] = calc_metrics(model, target_ds, 2)

metrics_x2.index = [
    "31px -> 62px", "63px -> 126px", "127px -> 254px", 
    "255px -> 510px", "510px -> 1020px", "1020px -> 2040px"
]
metrics_x2

In [ ]:
model_gse = GSE(model)

In [ ]:
targets = [X32_valid_paths, X16_valid_paths, X8_valid_paths, X4_valid_paths, X2_valid_paths, HR_valid_paths]

metrics_x2_gse = pd.DataFrame(columns=["PSNR↑", "SSIM↑", "LPIPS↓"])
for target_ds in tqdm(targets, total=6):
    metrics_x2_gse.loc[len(metrics_x2_gse)] = calc_metrics(model_gse, target_ds, 2)

metrics_x2_gse.index = [
    "31px -> 62px", "63px -> 126px", "127px -> 254px", 
    "255px -> 510px", "510px -> 1020px", "1020px -> 2040px"
]
metrics_x2_gse

In [ ]:
index_x2 = pd.MultiIndex.from_product([["X2"], [
    "31px -> 62px", "63px -> 126px", "127px -> 254px", 
    "255px -> 510px", "510px -> 1020px", "1020px -> 2040px"
]], names=["Scaling", "Resolution"])

index_x4 = pd.MultiIndex.from_product([["X4"], [
    "31px -> 124px", "63px -> 252px", "127px -> 508px", 
    "255px -> 1020px", "510px -> 2040px"
]], names=["Scaling", "Resolution"])

index_x8 = pd.MultiIndex.from_product([["X8"], [
    "31px -> 248px", "63px -> 504px", "127px -> 1016px", "255px -> 2040px"]], names=["Scaling", "Resolution"])

metrics_x2.index = index_x2
metrics_x4.index = index_x4
metrics_x8.index = index_x8

empty_row = pd.DataFrame([[np.nan, np.nan, np.nan]], columns=["PSNR↑", "SSIM↑", "LPIPS↓"], 
                         index = pd.MultiIndex.from_tuples([("--", "--")], names=["Scaling", "Resolution"]))
metrics_full = pd.concat([metrics_x2, empty_row, metrics_x4, empty_row, metrics_x8])
metrics_full

In [ ]:
index_x2 = pd.MultiIndex.from_product([["X2"], [
    "31px -> 62px", "63px -> 126px", "127px -> 254px", 
    "255px -> 510px", "510px -> 1020px", "1020px -> 2040px"
]], names=["Scaling", "Resolution"])

index_x4 = pd.MultiIndex.from_product([["X4"], [
    "31px -> 124px", "63px -> 252px", "127px -> 508px", 
    "255px -> 1020px", "510px -> 2040px"
]], names=["Scaling", "Resolution"])

index_x8 = pd.MultiIndex.from_product([["X8"], [
    "31px -> 248px", "63px -> 504px", "127px -> 1016px", "255px -> 2040px"]], names=["Scaling", "Resolution"])

metrics_x2_gse.index = index_x2
metrics_x4_gse.index = index_x4
metrics_x8_gse.index = index_x8

empty_row = pd.DataFrame([[np.nan, np.nan, np.nan]], columns=["PSNR↑", "SSIM↑", "LPIPS↓"], 
                         index = pd.MultiIndex.from_tuples([("--", "--")], names=["Scaling", "Resolution"]))
metrics_full_gse = pd.concat([metrics_x2_gse, empty_row, metrics_x4_gse, empty_row, metrics_x8_gse])
metrics_full_gse

### Save the results for later comparison

In [ ]:
os.makedirs('metrics_results', exist_ok=True)
metrics_full.to_parquet("metrics_results/edsr_light_results.parquet", engine="pyarrow")
metrics_full_gse.to_parquet("metrics_results/edsr_light+_results.parquet", engine="pyarrow")

In [ ]:
df = pd.read_parquet("metrics_results/edsr_light_results.parquet")
df

In [ ]:
df = pd.read_parquet("metrics_results/edsr_light+_results.parquet")
df